# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# constants

MODEL_GPT = 'gpt-6-luna'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# imports and environment setup

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from enum import Enum
from openai import OpenAI

# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-6-luna'
# openai = OpenAI()

In [ ]:
# setup client and call to client

class Provider(Enum):
    OPENAI = "OpenAI GPT 6 Luna "
    OLLAMA = "Llama 3.2"

def create_client(provider: Provider) -> OpenAI:
    match provider:
        case Provider.OPENAI:
            return OpenAI()
        case Provider.OLLAMA:
            return OpenAI(
                base_url=OLLAMA_BASE_URL,
                api_key="ollama"
            )
        case _:
            raise ValueError(f"Unsupported provider: {provider}")

def call_client(provider: Provider, system_prompt, user_prompt):
    client = create_client(provider)
    messages = [
                    { "role" : "system", "content" : system_prompt },
                    { "role" : "user", "content" : user_prompt }
                ]
    match provider:
        case Provider.OPENAI:
            return client.chat.completions.create(
                model=MODEL_GPT,
                reasoning_effort="none",
                messages=messages,
                stream=True
            )
        case Provider.OLLAMA:
            return client.chat.completions.create(
                model=MODEL_LLAMA,
                messages=messages,
                stream=True
            )
        case _:
            raise ValueError(f"Unsupported provider: {provider}")
            

In [ ]:
# setup system and user prompts, call client and stream response

def explain(concept, source, target, provider:Provider=Provider.OPENAI):
    system_prompt = """
        You are a programming tutor who is helping an experienced coder learn a new target
        language using their knowledge of a source language.
        
        Explain the requested concept in the target language, relating it to familar concepts
        in the source language where useful.

        Provide a brief explanation and 1-2 short target language code examples.
        Highlight important differences. If there is no direct equivalent, say so instead of forcing 
        an analogy.

        Assume general programming knowledge and focus on language-specific details.
    """

    user_prompt = f"""
        I want to understand how {concept} works in {target}. I am proficient and 
        have relevant work experience using {source} at a professional level. I'm trying to
        learn {target} using my understanding of {source}.
    """

    stream = call_client(
                provider=provider,
                system_prompt=system_prompt,
                user_prompt=user_prompt
            )

    response = f"### Model: {provider.value.upper()}\n\n"
    output = display(Markdown(""), display_id=True)


    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        output.update(Markdown(response))
    return response

In [ ]:
gpt_answer = explain(
    concept="switch cases",
    source="Swift",
    target="Python",
    provider=Provider.OPENAI,
)

ollama_answer = explain(
    concept="switch cases",
    source="Swift",
    target="Python",
    provider=Provider.OLLAMA,
)